# TeleGuard Churn Engine - Exploratory Data Analysis
This notebook performs initial EDA on the raw Telco dataset to uncover key drivers of customer churn.

### 1. Imports and Data Loading
Load `telco_raw.csv`, format the columns, drop missing values, and display the baseline metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style for plots
sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv('../data/telco_raw.csv')

# Convert TotalCharges to numeric and drop NaNs
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Print shape and churn rate
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"Overall Churn Rate: {churn_rate:.2f}%")

### 2. Churn Rate by Contract Type
Analyzing how contract lengths affect customer retention.

In [ ]:
os.makedirs('../screenshots', exist_ok=True)

# Calculate churn percentage by contract type
df['Churn_Numeric'] = df['Churn'].map({'Yes': 1, 'No': 0})
contract_churn = df.groupby('Contract')['Churn_Numeric'].mean() * 100

# Plotting
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=contract_churn.index, y=contract_churn.values, hue=contract_churn.index, palette="Blues_d", legend=False)
plt.title('Churn Rate by Contract Type', fontsize=14, pad=15)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.xlabel('Contract Type', fontsize=12)
plt.ylim(0, 60)

# Add percentage labels on top of bars
for i, v in enumerate(contract_churn.values):
    ax.text(i, v + 1, f"{v:.2f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/churn_by_contract.png', dpi=150)
plt.show()

print("Churn percentages by Contract Type:")
print(contract_churn.apply(lambda x: f"{x:.2f}%"))

### 3. Churn Rate by Tenure
Grouping customers by how many months they have been with the company to spot early-churn trends.

In [ ]:
# Create tenure buckets
bins = [0, 12, 24, 48, 72]
labels = ['0-12 months', '13-24 months', '25-48 months', '49-72 months']
df['Tenure_Bucket'] = pd.cut(df['tenure'], bins=bins, labels=labels, right=True)

# Calculate churn percentage by bucket
tenure_churn = df.groupby('Tenure_Bucket', observed=False)['Churn_Numeric'].mean() * 100

# Plotting
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=tenure_churn.index, y=tenure_churn.values, hue=tenure_churn.index, palette="Reds_r", legend=False)
plt.title('Churn Rate by Tenure Bucket', fontsize=14, pad=15)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.xlabel('Tenure Bucket', fontsize=12)
plt.ylim(0, 65)

# Add percentage labels
for i, v in enumerate(tenure_churn.values):
    ax.text(i, v + 1, f"{v:.2f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/churn_by_tenure.png', dpi=150)
plt.show()

print("Churn percentages by Tenure Bucket:")
print(tenure_churn.apply(lambda x: f"{x:.2f}%"))

### 4. Monthly Charges Distribution
Comparing the cost distribution between churned and retained customers.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(data=df[df['Churn'] == 'No'], x='MonthlyCharges', color='green', label='Retained (No)', kde=True, alpha=0.5)
sns.histplot(data=df[df['Churn'] == 'Yes'], x='MonthlyCharges', color='red', label='Churned (Yes)', kde=True, alpha=0.5)

plt.title('Monthly Charges Distribution: Retained vs Churned Customers', fontsize=14, pad=15)
plt.xlabel('Monthly Charges ($)', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.legend()

plt.tight_layout()
plt.savefig('../screenshots/charges_distribution.png', dpi=150)
plt.show()

### 5. Top 10 Features Correlated with Churn
Extracting the most predictive features using correlation metrics.

In [ ]:
# Create a copy for correlation encoding
corr_df = df.copy()
corr_df.drop(['Tenure_Bucket', 'customerID'], axis=1, inplace=True, errors='ignore')

# Convert binary columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if col in corr_df.columns:
        corr_df[col] = corr_df[col].map({'Yes': 1, 'No': 0})

# Get dummies for remaining categorical columns
cat_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 
            'StreamingMovies', 'Contract', 'PaymentMethod']
cat_cols = [c for c in cat_cols if c in corr_df.columns]
corr_df = pd.get_dummies(corr_df, columns=cat_cols, drop_first=True)

# Ensure numeric types
numeric_df = corr_df.select_dtypes(include=[np.number])

# Calculate correlations with Churn_Numeric
correlations = numeric_df.corr()['Churn_Numeric'].drop('Churn_Numeric')

# Get top 10 absolute correlations
top_10_corr = correlations.abs().sort_values(ascending=False).head(10)
top_10_features = correlations[top_10_corr.index]

# Plotting
plt.figure(figsize=(10, 6))
colors = ['#d62728' if c > 0 else '#2ca02c' for c in top_10_features.values]
ax = sns.barplot(x=top_10_features.values, y=top_10_features.index, hue=top_10_features.index, palette=colors, legend=False)

plt.title('Top 10 Features Most Correlated with Churn', fontsize=14, pad=15)
plt.xlabel('Correlation Coefficient', fontsize=12)
plt.ylabel('Feature', fontsize=12)

# Add value labels
for i, v in enumerate(top_10_features.values):
    offset = 0.02 if v > 0 else -0.02
    ha = 'left' if v > 0 else 'right'
    ax.text(v + offset, i, f"{v:.3f}", va='center', ha=ha, fontweight='bold')

# Adjust x-axis to fit labels
plt.xlim(min(top_10_features.values) - 0.1, max(top_10_features.values) + 0.1)
plt.tight_layout()
plt.savefig('../screenshots/top_features.png', dpi=150)
plt.show()

### 6. Executive Summary
A clean output of the most critical numbers for the presentation.

In [ ]:
m2m_rate = contract_churn.get('Month-to-month', 0)
two_year_rate = contract_churn.get('Two year', 0)
first_year_rate = tenure_churn.get('0-12 months', 0)
top_feature = top_10_features.index[0]

print("=== TELEGUARD KEY FINDINGS ===")
print(f"Month-to-month churn rate: {m2m_rate:.2f}%{' ':>22}Two-year contract churn rate: {two_year_rate:.2f}%")
print(f"First-year churn rate: {first_year_rate:.2f}%")
print(f"Most correlated feature: {top_feature}")